In [10]:
#3
import math

class TicTacToeMinimax:
    def __init__(self, size=5, win_condition=4):
        self.size = size
        self.win_req = win_condition
        self.board = [[' ' for _ in range(size)] for _ in range(size)]

    def evaluate_board(self):
        """
        Hàm lượng giá: Trả về điểm số dựa trên trạng thái bàn cờ hiện tại.
        Dành cho ma trận lớn, tính toán số lượng chuỗi quân cờ.
        """


        if self._check_win('X'):
            return 1000

        if self._check_win('O'):
            return -1000

        return 0

    def _check_win(self, player):

        for r in range(self.size):
            for c in range(self.size - self.win_req + 1):
                if all(self.board[r][c+i] == player for i in range(self.win_req)):
                    return True

        for c in range(self.size):
            for r in range(self.size - self.win_req + 1):
                if all(self.board[r+i][c] == player for i in range(self.win_req)):
                    return True

        for r in range(self.size - self.win_req + 1):
            for c in range(self.size - self.win_req + 1):
                if all(self.board[r+i][c+i] == player for i in range(self.win_req)):
                    return True

        for r in range(self.size - self.win_req + 1):
            for c in range(self.win_req - 1, self.size):
                if all(self.board[r+i][c-i] == player for i in range(self.win_req)):
                    return True
        return False

    def is_game_over(self):

        if self._check_win('X') or self._check_win('O'):
            return True

        if not self.get_empty_cells():
            return True
        return False

    def minimax(self, depth, alpha, beta, is_maximizing):

        if depth == 0 or self.is_game_over():
            return self.evaluate_board()

        if is_maximizing:
            max_eval = -math.inf
            for r, c in self.get_empty_cells():
                self.board[r][c] = 'X'
                eval = self.minimax(depth - 1, alpha, beta, False)
                self.board[r][c] = ' '
                max_eval = max(max_eval, eval)
                alpha = max(alpha, eval)
                if beta <= alpha:
                    break
            return max_eval
        else:
            min_eval = math.inf
            for r, c in self.get_empty_cells():
                self.board[r][c] = 'O'
                eval = self.minimax(depth - 1, alpha, beta, True)
                self.board[r][c] = ' '
                min_eval = min(min_eval, eval)
                beta = min(beta, eval)
                if beta <= alpha:
                    break
            return min_eval

    def get_empty_cells(self):

        cells = []
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] == ' ':
                    cells.append((r, c))
        return cells

    def print_board(self):
        for row in self.board:
            print('| ' + ' | '.join(row) + ' |')
            print('-' * (4 * self.size + 1))

    def find_best_move(self, depth):
        best_eval = -math.inf
        best_move = None
        alpha = -math.inf
        beta = math.inf

        for r, c in self.get_empty_cells():
            self.board[r][c] = 'X'
            eval = self.minimax(depth - 1, alpha, beta, False)
            self.board[r][c] = ' '
            if eval > best_eval:
                best_eval = eval
                best_move = (r, c)
            alpha = max(alpha, eval)
        return best_move

def main():
    game = TicTacToeMinimax(size=5, win_condition=4)
    current_player = 'O'

    while not game.is_game_over():
        game.print_board()

        if current_player == 'O':
            print("Người chơi (O) di chuyển.")
            try:
                row = int(input("Nhập hàng (0-4): "))
                col = int(input("Nhập cột (0-4): "))
                if not (0 <= row < game.size and 0 <= col < game.size and game.board[row][col] == ' '):
                    print("Nước đi không hợp lệ. Vui lòng thử lại.")
                    continue
                game.board[row][col] = 'O'
            except ValueError:
                print("Đầu vào không hợp lệ. Vui lòng nhập số.")
                continue
        else:
            print("AI (X) di chuyển...")

            ai_move = game.find_best_move(depth=3)
            if ai_move:
                game.board[ai_move[0]][ai_move[1]] = 'X'
            else:

                empty_cells = game.get_empty_cells()
                if empty_cells:
                    r, c = empty_cells[0]
                    game.board[r][c] = 'X'

        if game.is_game_over():
            break

        current_player = 'X' if current_player == 'O' else 'O'

    game.print_board()
    if game._check_win('X'):
        print("AI (X) thắng!")
    elif game._check_win('O'):
        print("Người chơi (O) thắng!")
    else:
        print("Hòa!")

if __name__ == '__main__':
    main()

|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
Người chơi (O) di chuyển.
Nhập hàng (0-4): 1
Nhập cột (0-4): 2
|   |   |   |   |   |
---------------------
|   |   | O |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
AI (X) di chuyển...
| X |   |   |   |   |
---------------------
|   |   | O |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
Người chơi (O) di chuyển.


KeyboardInterrupt: Interrupted by user

In [16]:
import math

class TicTacToeAI:
    def __init__(self, size, win_condition=5):
        self.size = size
        self.win_req = win_condition
        self.board = [[' ' for _ in range(size)] for _ in range(size)]

    def evaluate_line(self, line, player):
        """Tính điểm cho một hàng/cột/chéo"""
        score = 0
        opp = 'O' if player == 'X' else 'X'

        count_player = line.count(player)
        count_opp = line.count(opp)

        if count_player == self.win_req: return 10000
        if count_opp == self.win_req: return -10000

        return count_player ** 2

    def evaluate_board(self):
        """Tổng hợp điểm của toàn bộ bàn cờ"""
        total_score = 0

        # Đánh giá hàng ngang
        for r in range(self.size):
            for c in range(self.size - self.win_req + 1):
                line = [self.board[r][c+i] for i in range(self.win_req)]
                total_score += self.evaluate_line(line, 'X')  # Đánh giá cho AI
                total_score -= self.evaluate_line(line, 'O')  # Trừ điểm nếu có lợi cho đối thủ

        # Đánh giá hàng dọc
        for c in range(self.size):
            for r in range(self.size - self.win_req + 1):
                line = [self.board[r+i][c] for i in range(self.win_req)]
                total_score += self.evaluate_line(line, 'X')
                total_score -= self.evaluate_line(line, 'O')

        # Đánh giá đường chéo chính (từ trên trái xuống dưới phải)
        for r in range(self.size - self.win_req + 1):
            for c in range(self.size - self.win_req + 1):
                line = [self.board[r+i][c+i] for i in range(self.win_req)]
                total_score += self.evaluate_line(line, 'X')
                total_score -= self.evaluate_line(line, 'O')

        # Đánh giá đường chéo phụ (từ trên phải xuống dưới trái)
        for r in range(self.size - self.win_req + 1):
            for c in range(self.win_req - 1, self.size):
                line = [self.board[r+i][c-i] for i in range(self.win_req)]
                total_score += self.evaluate_line(line, 'X')
                total_score -= self.evaluate_line(line, 'O')

        return total_score

    def _check_win(self, player):
        """Kiểm tra xem người chơi đã thắng chưa"""
        # Check rows
        for r in range(self.size):
            for c in range(self.size - self.win_req + 1):
                if all(self.board[r][c+i] == player for i in range(self.win_req)):
                    return True
        # Check columns
        for c in range(self.size):
            for r in range(self.size - self.win_req + 1):
                if all(self.board[r+i][c] == player for i in range(self.win_req)):
                    return True
        # Check diagonals (top-left to bottom-right)
        for r in range(self.size - self.win_req + 1):
            for c in range(self.size - self.win_req + 1):
                if all(self.board[r+i][c+i] == player for i in range(self.win_req)):
                    return True
        # Check anti-diagonals (top-right to bottom-left)
        for r in range(self.size - self.win_req + 1):
            for c in range(self.win_req - 1, self.size):
                if all(self.board[r+i][c-i] == player for i in range(self.win_req)):
                    return True
        return False

    def is_game_over(self):
        """Kiểm tra xem trò chơi đã kết thúc chưa"""
        if self._check_win('X') or self._check_win('O'):
            return True
        if not self.get_empty_cells(): # If no empty cells, it's a draw
            return True
        return False

    def get_empty_cells(self):
        """Trả về danh sách các ô trống trên bàn cờ"""
        cells = []
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] == ' ':
                    cells.append((r, c))
        return cells

    def print_board(self):
        """In bàn cờ ra màn hình"""
        for row in self.board:
            print('| ' + ' | '.join(row) + ' |')
            print('-' * (4 * self.size + 1))

    def alpha_beta(self, depth, alpha, beta, is_maximizing):

        if depth == 0 or self.is_game_over():
            # Return the evaluation from the perspective of the maximizing player (AI 'X')
            return self.evaluate_board()

        moves = self.get_empty_cells()
        if not moves: return 0

        if is_maximizing:
            max_eval = -math.inf
            for r, c in moves:
                self.board[r][c] = 'X'
                eval = self.alpha_beta(depth - 1, alpha, beta, False)
                self.board[r][c] = ' '
                max_eval = max(max_eval, eval)
                alpha = max(alpha, eval)
                if beta <= alpha:
                    break
            return max_eval
        else:
            min_eval = math.inf
            for r, c in moves:
                self.board[r][c] = 'O'
                eval = self.alpha_beta(depth - 1, alpha, beta, True)
                self.board[r][c] = ' '
                min_eval = min(min_eval, eval)
                beta = min(beta, eval)
                if beta <= alpha:
                    break
            return min_eval

    def find_best_move(self, depth):
        """Tìm nước đi tốt nhất cho AI (người chơi 'X')"""
        best_eval = -math.inf
        best_move = None
        alpha = -math.inf
        beta = math.inf

        for r, c in self.get_empty_cells():
            self.board[r][c] = 'X'
            eval = self.alpha_beta(depth - 1, alpha, beta, False)
            self.board[r][c] = ' '
            if eval > best_eval:
                best_eval = eval
                best_move = (r, c)
            alpha = max(alpha, eval)
        return best_move

def main_tictactoeai():
    game = TicTacToeAI(size=5, win_condition=4)
    current_player = 'O'
    search_depth = 3 # Độ sâu tìm kiếm cho AI

    while not game.is_game_over():
        game.print_board()

        if current_player == 'O':
            print("Người chơi (O) di chuyển.")
            try:
                row = int(input(f"Nhập hàng (0-{game.size-1}): "))
                col = int(input(f"Nhập cột (0-{game.size-1}): "))
                if not (0 <= row < game.size and 0 <= col < game.size and game.board[row][col] == ' '):
                    print("Nước đi không hợp lệ. Vui lòng thử lại.")
                    continue
                game.board[row][col] = 'O'
            except ValueError:
                print("Đầu vào không hợp lệ. Vui lòng nhập số.")
                continue
        else:
            print("AI (X) di chuyển...")
            ai_move = game.find_best_move(depth=search_depth)
            if ai_move:
                game.board[ai_move[0]][ai_move[1]] = 'X'
            else:
                # Fallback if no best move is found (should not happen with proper minimax)
                empty_cells = game.get_empty_cells()
                if empty_cells:
                    r, c = empty_cells[0]
                    game.board[r][c] = 'X'

        if game.is_game_over():
            break

        current_player = 'X' if current_player == 'O' else 'O'

    game.print_board()
    if game._check_win('X'):
        print("AI (X) thắng!")
    elif game._check_win('O'):
        print("Người chơi (O) thắng!")
    else:
        print("Hòa!")

if __name__ == '__main__':
    main_tictactoeai()

|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
|   |   |   |   |   |
---------------------
Người chơi (O) di chuyển.



KeyboardInterrupt

